In [1]:
# =============================================================================
# CELL 1 — Imports & Configuration
# =============================================================================
# Purpose : Import all required libraries and define global configuration
#           for the parallel Silver cleansing pipeline.
#
# Parallelism Strategy:
#   - ThreadPoolExecutor runs each table's cleanse+write in a separate thread
#   - Spark is thread-safe for concurrent DataFrame operations
#   - MAX_WORKERS = 5 : one thread per Bronze table
#   - All 5 tables processed simultaneously — total time ≈ slowest single table
#
# Spark Optimizations applied across all tables:
#   - Native functions only (no UDFs) — Catalyst-safe
#   - Predicate pushdown via KQL time filter before Spark ingestion
#   - repartition() aligned to downstream Gold query patterns
#   - partitionBy(year, month, day) at write — right-sized for data volume
#   - Idempotent MERGE — safe for reruns and pipeline retries
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor, as_completed
import traceback

# -----------------------------------------------------------------------------
# KQL Source Config
# -----------------------------------------------------------------------------
KUSTO_CLUSTER  = "https://trd-ratdj1p1b0yurnmn41.z4.kusto.fabric.microsoft.com"
KUSTO_DATABASE = "pulsegrid_bronze"

# -----------------------------------------------------------------------------
# Silver Delta Table Paths (inside pulsegrid_lakehouse)
# -----------------------------------------------------------------------------
SILVER_PATHS = {
    "prices"    : "Tables/silver_electricity_prices",
    "load"      : "Tables/silver_electricity_load",
    "generation": "Tables/silver_generation_mix",
    "flows"     : "Tables/silver_cross_border_flows",
    "weather"   : "Tables/silver_weather"
}

# -----------------------------------------------------------------------------
# Parallel Execution Config
# -----------------------------------------------------------------------------
MAX_WORKERS = 5   # One thread per Bronze table

print("✅ Imports and config loaded")
print(f"   Tables to process : {len(SILVER_PATHS)}")
print(f"   Parallel workers  : {MAX_WORKERS}")
print(f"   KQL source        : {KUSTO_DATABASE}")

StatementMeta(, b2bf4225-c8a6-40b9-8331-6021a35983d2, 3, Finished, Available, Finished, False)

✅ Imports and config loaded
   Tables to process : 5
   Parallel workers  : 5
   KQL source        : pulsegrid_bronze


In [2]:
# =============================================================================
# CELL 2 — Generic KQL Bronze Reader
# =============================================================================
# Purpose : Read any Bronze KQL table into a Spark DataFrame.
#
# Spark Optimization — Predicate Pushdown:
#   - KQL filter (where ingestion_time > ago(7d)) executes on the KQL engine
#     BEFORE data is pulled into Spark.
#   - Reduces data transferred across the wire significantly.
#   - Trial-friendly: limits Spark memory pressure.
#
# Auth    : mssparkutils managed identity token — no hardcoded secrets.
# =============================================================================

def read_bronze_table(table_name, kql_filter="ingestion_time > ago(7d)"):
    """
    Read a Bronze KQL table into a Spark DataFrame.

    Args:
        table_name : KQL table name in pulsegrid_bronze
        kql_filter : KQL where clause for predicate pushdown (default: last 7 days)

    Returns:
        Spark DataFrame with raw Bronze data
    """
    query = f"{table_name} | where {kql_filter}"

    df = spark.read \
        .format("com.microsoft.kusto.spark.datasource") \
        .option("kustoCluster",  KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DATABASE) \
        .option("kustoQuery",    query) \
        .option("accessToken",   mssparkutils.credentials.getToken("kusto")) \
        .load()

    return df

print("✅ Generic KQL reader defined")

StatementMeta(, b2bf4225-c8a6-40b9-8331-6021a35983d2, 4, Finished, Available, Finished, False)

✅ Generic KQL reader defined


In [3]:
# =============================================================================
# CELL 3 — Table-Specific Cleansing Functions
# =============================================================================
# Purpose : Apply business rules and Spark optimizations to each Bronze table.
#
# Spark Optimization — Native Functions Only (No UDFs):
#   - F.when, F.coalesce, F.upper, F.trim, F.to_timestamp used throughout
#   - Native functions are visible to Catalyst optimizer
#   - UDFs are opaque to Catalyst — incur Python↔JVM serialization per row
#
# Deduplication Strategy:
#   - Window function row_number() over natural key, ordered by ingestion_time DESC
#   - Keeps most recently ingested record — deterministic, fully distributed
#   - No collect() to driver
#
# Partition Columns:
#   - year/month/day added to all tables for Delta partition pruning
#   - repartition() by region + hour aligns Spark partitions with Gold patterns
# =============================================================================

# -----------------------------------------------------------------------------
# Helper — Add partition columns + repartition
# -----------------------------------------------------------------------------
def add_partitions_and_repartition(df, time_col="event_time"):
    """Add year/month/day partition columns and repartition by region + hour."""
    return df \
        .withColumn("year",  F.year(F.col(time_col))) \
        .withColumn("month", F.month(F.col(time_col))) \
        .withColumn("day",   F.dayofmonth(F.col(time_col))) \
        .repartition(F.col("region"), F.hour(F.col(time_col)))

# -----------------------------------------------------------------------------
# Helper — Deduplicate by natural key
# -----------------------------------------------------------------------------
def deduplicate(df, partition_cols):
    """
    Remove duplicates keeping the latest ingested record.
    partition_cols: list of columns forming the natural key
    """
    window = Window \
        .partitionBy(*partition_cols) \
        .orderBy(F.col("ingestion_time").desc())

    return df \
        .withColumn("row_num", F.row_number().over(window)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num")

# -----------------------------------------------------------------------------
# 1. Electricity Prices Cleansing
# -----------------------------------------------------------------------------
def cleanse_prices(df):
    """
    Cleanse raw_electricity_prices:
    - Nullify prices outside valid range [-500, 5000] EUR/MWh
      (European markets allow negative prices during oversupply)
    - Nullify negative load values (physically impossible)
    - Normalize region to uppercase
    - Deduplicate on (region, event_time)
    """
    df_clean = df \
        .withColumn("price_eur_mwh",
            F.when(
                (F.col("price_eur_mwh") < -500) | (F.col("price_eur_mwh") > 5000),
                F.lit(None)
            ).otherwise(F.col("price_eur_mwh"))
        ) \
        .withColumn("load_mw",
            F.when(F.col("load_mw") < 0, F.lit(None))
            .otherwise(F.col("load_mw"))
        ) \
        .withColumn("temperature_c",
            F.coalesce(F.col("temperature_c"), F.lit(0.0))
        ) \
        .withColumn("region", F.upper(F.trim(F.col("region")))) \
        .withColumn("event_time", F.to_timestamp(F.col("event_time"))) \
        .withColumn("ingestion_time", F.to_timestamp(F.col("ingestion_time")))

    df_deduped = deduplicate(df_clean, ["region", "event_time"])
    return add_partitions_and_repartition(df_deduped)

# -----------------------------------------------------------------------------
# 2. Electricity Load Cleansing
# -----------------------------------------------------------------------------
def cleanse_load(df):
    """
    Cleanse raw_electricity_load:
    - Nullify negative or zero load values
    - Cap extreme values > 1,000,000 MW (data error)
    - Normalize region, deduplicate on (region, event_time)
    """
    df_clean = df \
        .withColumn("load_mw",
            F.when(
                (F.col("load_mw") <= 0) | (F.col("load_mw") > 1000000),
                F.lit(None)
            ).otherwise(F.col("load_mw"))
        ) \
        .withColumn("region", F.upper(F.trim(F.col("region")))) \
        .withColumn("event_time", F.to_timestamp(F.col("event_time"))) \
        .withColumn("ingestion_time", F.to_timestamp(F.col("ingestion_time")))

    df_deduped = deduplicate(df_clean, ["region", "event_time"])
    return add_partitions_and_repartition(df_deduped)

# -----------------------------------------------------------------------------
# 3. Generation Mix Cleansing
# -----------------------------------------------------------------------------
def cleanse_generation(df):
    """
    Cleanse raw_generation_mix:
    - Nullify negative generation values
    - Normalize fuel_type to title case (e.g. 'wind onshore' → 'Wind Onshore')
    - Normalize region, deduplicate on (region, event_time, fuel_type)
    """
    df_clean = df \
        .withColumn("generation_mw",
            F.when(F.col("generation_mw") < 0, F.lit(None))
            .otherwise(F.col("generation_mw"))
        ) \
        .withColumn("fuel_type",
            F.initcap(F.trim(F.col("fuel_type")))
        ) \
        .withColumn("region", F.upper(F.trim(F.col("region")))) \
        .withColumn("event_time", F.to_timestamp(F.col("event_time"))) \
        .withColumn("ingestion_time", F.to_timestamp(F.col("ingestion_time")))

    df_deduped = deduplicate(df_clean, ["region", "event_time", "fuel_type"])

    return df_deduped \
        .withColumn("year",  F.year(F.col("event_time"))) \
        .withColumn("month", F.month(F.col("event_time"))) \
        .withColumn("day",   F.dayofmonth(F.col("event_time"))) \
        .repartition(F.col("region"), F.col("fuel_type"))

# -----------------------------------------------------------------------------
# 4. Cross-Border Flows Cleansing
# -----------------------------------------------------------------------------
def cleanse_flows(df):
    """
    Cleanse raw_cross_border_flows:
    - Normalize from_region and to_region to uppercase
    - Remove self-flows (from_region == to_region — data error)
    - Deduplicate on (from_region, to_region, event_time)
    - Note: negative flow_mw is valid (import direction)
    """
    df_clean = df \
        .withColumn("from_region", F.upper(F.trim(F.col("from_region")))) \
        .withColumn("to_region",   F.upper(F.trim(F.col("to_region")))) \
        .withColumn("event_time",     F.to_timestamp(F.col("event_time"))) \
        .withColumn("ingestion_time", F.to_timestamp(F.col("ingestion_time"))) \
        .filter(F.col("from_region") != F.col("to_region"))

    df_deduped = deduplicate(df_clean, ["from_region", "to_region", "event_time"])

    return df_deduped \
        .withColumn("year",  F.year(F.col("event_time"))) \
        .withColumn("month", F.month(F.col("event_time"))) \
        .withColumn("day",   F.dayofmonth(F.col("event_time"))) \
        .repartition(F.col("from_region"), F.col("to_region"))

# -----------------------------------------------------------------------------
# 5. Weather Cleansing
# -----------------------------------------------------------------------------
def cleanse_weather(df):
    """
    Cleanse raw_weather:
    - Temperature: valid range [-60, 60] °C
    - Wind speed: non-negative, cap at 100 m/s
    - Humidity: valid range [0, 100] %
    - Solar radiation: non-negative, cap at 1500 W/m²
    - Fill nulls with 0.0 for all numeric fields (sensor outage fallback)
    - Deduplicate on (region, event_time)
    """
    df_clean = df \
        .withColumn("temperature_c",
            F.when(
                (F.col("temperature_c") < -60) | (F.col("temperature_c") > 60),
                F.lit(None)
            ).otherwise(F.col("temperature_c"))
        ) \
        .withColumn("wind_speed_ms",
            F.when(
                (F.col("wind_speed_ms") < 0) | (F.col("wind_speed_ms") > 100),
                F.lit(None)
            ).otherwise(F.col("wind_speed_ms"))
        ) \
        .withColumn("humidity_pct",
            F.when(
                (F.col("humidity_pct") < 0) | (F.col("humidity_pct") > 100),
                F.lit(None)
            ).otherwise(F.col("humidity_pct"))
        ) \
        .withColumn("solar_radiation",
            F.when(
                (F.col("solar_radiation") < 0) | (F.col("solar_radiation") > 1500),
                F.lit(None)
            ).otherwise(F.col("solar_radiation"))
        ) \
        .withColumn("temperature_c",   F.coalesce(F.col("temperature_c"),   F.lit(0.0))) \
        .withColumn("wind_speed_ms",   F.coalesce(F.col("wind_speed_ms"),   F.lit(0.0))) \
        .withColumn("humidity_pct",    F.coalesce(F.col("humidity_pct"),    F.lit(0.0))) \
        .withColumn("solar_radiation", F.coalesce(F.col("solar_radiation"), F.lit(0.0))) \
        .withColumn("region", F.upper(F.trim(F.col("region")))) \
        .withColumn("event_time",     F.to_timestamp(F.col("event_time"))) \
        .withColumn("ingestion_time", F.to_timestamp(F.col("ingestion_time")))

    df_deduped = deduplicate(df_clean, ["region", "event_time"])
    return add_partitions_and_repartition(df_deduped)

print("✅ All cleansing functions defined")

StatementMeta(, b2bf4225-c8a6-40b9-8331-6021a35983d2, 5, Finished, Available, Finished, False)

✅ All cleansing functions defined


In [4]:
# =============================================================================
# CELL 4 — Generic Silver Delta Writer (Idempotent MERGE)
# =============================================================================
# Purpose : Write any cleansed DataFrame to its Silver Delta table.
#           Uses MERGE for idempotent, rerun-safe writes.
#
# MERGE Strategy:
#   - First run  : DeltaTable.isDeltaTable() = False → full write (creates table)
#   - Subsequent : MERGE on natural key → update existing + insert new
#   - Safe for reruns, pipeline retries, and backfill scenarios
#
# merge_keys : columns forming the natural key for MERGE condition
#              Must uniquely identify one record in the target table
# =============================================================================

def write_silver(df, silver_path, merge_keys):
    """
    Write a cleansed DataFrame to Silver Delta table using idempotent MERGE.

    Args:
        df          : Cleansed Spark DataFrame
        silver_path : Delta table path in pulsegrid_lakehouse
        merge_keys  : List of column names forming the natural key
    """
    partition_cols = ["year", "month", "day"]

    if DeltaTable.isDeltaTable(spark, silver_path):
        # Build MERGE condition from natural keys
        merge_condition = " AND ".join(
            [f"target.{k} = source.{k}" for k in merge_keys]
        )

        delta_table = DeltaTable.forPath(spark, silver_path)
        delta_table.alias("target").merge(
            df.alias("source"),
            merge_condition
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    else:
        # First run — create the Silver Delta table
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .partitionBy(*partition_cols) \
            .option("mergeSchema", "false") \
            .save(silver_path)

print("✅ Generic Silver writer defined")

StatementMeta(, b2bf4225-c8a6-40b9-8331-6021a35983d2, 6, Finished, Available, Finished, False)

✅ Generic Silver writer defined


In [5]:
# =============================================================================
# CELL 5 — Parallel Silver Processing (ThreadPoolExecutor)
# =============================================================================
# Purpose : Process all 5 Bronze tables concurrently using Python threads.
#
# Why ThreadPoolExecutor works here:
#   - Spark DataFrame operations are thread-safe
#   - Each thread submits independent Spark jobs to the cluster
#   - Spark scheduler runs jobs concurrently on available executors
#   - Total wall time ≈ slowest single table (not sum of all tables)
#
# Task map : defines Bronze table → cleanse function → Silver path → merge keys
#   Each entry is a self-contained pipeline for one table.
#
# Error handling:
#   - Each task wrapped in try/except — one table failure doesn't
#     block other tables from completing
#   - Results logged per task with success/failure status
# =============================================================================

# Task map — one entry per Bronze table
SILVER_TASKS = [
    {
        "name"       : "electricity_prices",
        "bronze_table" : "raw_electricity_prices",
        "cleanse_fn" : cleanse_prices,
        "silver_path": SILVER_PATHS["prices"],
        "merge_keys" : ["region", "event_time"]
    },
    {
        "name"       : "electricity_load",
        "bronze_table" : "raw_electricity_load",
        "cleanse_fn" : cleanse_load,
        "silver_path": SILVER_PATHS["load"],
        "merge_keys" : ["region", "event_time"]
    },
    {
        "name"       : "generation_mix",
        "bronze_table" : "raw_generation_mix",
        "cleanse_fn" : cleanse_generation,
        "silver_path": SILVER_PATHS["generation"],
        "merge_keys" : ["region", "event_time", "fuel_type"]
    },
    {
        "name"       : "cross_border_flows",
        "bronze_table" : "raw_cross_border_flows",
        "cleanse_fn" : cleanse_flows,
        "silver_path": SILVER_PATHS["flows"],
        "merge_keys" : ["from_region", "to_region", "event_time"]
    },
    {
        "name"       : "weather",
        "bronze_table" : "raw_weather",
        "cleanse_fn" : cleanse_weather,
        "silver_path": SILVER_PATHS["weather"],
        "merge_keys" : ["region", "event_time"]
    },
]


def process_table(task):
    """
    Full pipeline for one table:
    1. Read from Bronze KQL
    2. Cleanse
    3. Write to Silver Delta
    """
    name = task["name"]
    try:
        print(f"   🔄 [{name}] Starting...")

        # Step 1 — Read Bronze
        df_bronze = read_bronze_table(task["bronze_table"])
        bronze_count = df_bronze.count()
        print(f"   📥 [{name}] Bronze records: {bronze_count}")

        # Step 2 — Cleanse
        df_silver = task["cleanse_fn"](df_bronze)
        silver_count = df_silver.count()
        print(f"   🧹 [{name}] After cleansing: {silver_count}")

        # Step 3 — Write Silver
        write_silver(df_silver, task["silver_path"], task["merge_keys"])
        print(f"   ✅ [{name}] Written to Silver successfully")

        return {"name": name, "status": "success", "records": silver_count}

    except Exception as e:
        print(f"   ❌ [{name}] Failed: {e}")
        traceback.print_exc()
        return {"name": name, "status": "failed", "error": str(e)}


# -----------------------------------------------------------------------------
# Execute all tasks in parallel
# -----------------------------------------------------------------------------
print("=" * 55)
print("  PulseGrid Silver — Parallel Processing Start")
print("=" * 55)

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_table, task): task for task in SILVER_TASKS}
    for future in as_completed(futures):
        result = future.result()
        results.append(result)

print("\n" + "=" * 55)
print("  Parallel Processing Complete")
print("=" * 55)

StatementMeta(, b2bf4225-c8a6-40b9-8331-6021a35983d2, 7, Finished, Available, Finished, False)

  PulseGrid Silver — Parallel Processing Start
   🔄 [electricity_prices] Starting...
   🔄 [electricity_load] Starting...
   🔄 [generation_mix] Starting...
   🔄 [cross_border_flows] Starting...
   🔄 [weather] Starting...
   📥 [cross_border_flows] Bronze records: 90
   📥 [generation_mix] Bronze records: 1669
   📥 [electricity_load] Bronze records: 120
   📥 [weather] Bronze records: 20
   🧹 [electricity_load] After cleansing: 120
   🧹 [weather] After cleansing: 20
   🧹 [cross_border_flows] After cleansing: 90
   🧹 [generation_mix] After cleansing: 1669
   ✅ [generation_mix] Written to Silver successfully
   ✅ [weather] Written to Silver successfully
   ✅ [cross_border_flows] Written to Silver successfully
   ✅ [electricity_load] Written to Silver successfully
   📥 [electricity_prices] Bronze records: 36
   🧹 [electricity_prices] After cleansing: 36
   ✅ [electricity_prices] Written to Silver successfully

  Parallel Processing Complete


In [6]:
# =============================================================================
# CELL 6 — Silver Layer Validation
# =============================================================================
# Purpose : Read back all 5 Silver Delta tables and validate row counts,
#           null rates, and date coverage. Data quality gate before Gold.
# =============================================================================

print("=" * 55)
print("  Silver Layer — Validation Report")
print("=" * 55)

validation_map = {
    "silver_electricity_prices" : (SILVER_PATHS["prices"],     "price_eur_mwh"),
    "silver_electricity_load"   : (SILVER_PATHS["load"],       "load_mw"),
    "silver_generation_mix"     : (SILVER_PATHS["generation"],  "generation_mw"),
    "silver_cross_border_flows" : (SILVER_PATHS["flows"],       "flow_mw"),
    "silver_weather"            : (SILVER_PATHS["weather"],     "temperature_c"),
}

for table_name, (path, numeric_col) in validation_map.items():
    try:
        df = spark.read.format("delta").load(path)
        row_count  = df.count()
        null_count = df.filter(F.col(numeric_col).isNull()).count()
        date_range = df.agg(
            F.min("event_time").alias("earliest"),
            F.max("event_time").alias("latest")
        ).collect()[0]

        print(f"\n  📋 {table_name}")
        print(f"     Rows        : {row_count}")
        print(f"     Nulls ({numeric_col}) : {null_count}")
        print(f"     Date range  : {date_range['earliest']} → {date_range['latest']}")

    except Exception as e:
        print(f"\n  ❌ {table_name} — could not read: {e}")

print("\n" + "=" * 55)

# Summary table
print("\n  Processing Summary:")
print(f"  {'Table':<30} {'Status':<10} {'Records'}")
print(f"  {'-'*50}")
for r in results:
    status  = "✅ Success" if r["status"] == "success" else "❌ Failed"
    records = r.get("records", "N/A")
    print(f"  {r['name']:<30} {status:<10} {records}")
print("=" * 55)

StatementMeta(, b2bf4225-c8a6-40b9-8331-6021a35983d2, 8, Finished, Available, Finished, False)

  Silver Layer — Validation Report

  📋 silver_electricity_prices
     Rows        : 66
     Nulls (price_eur_mwh) : 36
     Date range  : 2026-08-13 06:00:00 → 2026-08-14 07:00:00

  📋 silver_electricity_load
     Rows        : 140
     Nulls (load_mw) : 1
     Date range  : 2026-08-13 06:00:00 → 2026-08-14 06:45:00

  📋 silver_generation_mix
     Rows        : 1689
     Nulls (generation_mw) : 0
     Date range  : 2026-08-13 06:00:00 → 2026-08-14 06:45:00

  📋 silver_cross_border_flows
     Rows        : 105
     Nulls (flow_mw) : 0
     Date range  : 2026-08-13 06:00:00 → 2026-08-14 06:45:00

  📋 silver_weather
     Rows        : 38
     Nulls (temperature_c) : 0
     Date range  : 2026-08-13 06:00:00 → 2026-08-14 08:29:40.065228


  Processing Summary:
  Table                          Status     Records
  --------------------------------------------------
  generation_mix                 ✅ Success  1669
  weather                        ✅ Success  20
  cross_border_flows            